# Stage 1 — F1..F5 Forensic Branch

Uses the team-provided fixed `train.csv` / `val.csv` directly.

Recommended order:
`bayar_resnet18 → chromaticity → frequency → lcdf → cdc`.

Native-resolution frames are cropped before any resize; final scoring is video-level Macro-F1.

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

try:
    import blackbox_detection  # noqa: F401
except ModuleNotFoundError:
    _root = Path.cwd()
    while _root != _root.parent and not (_root / "pyproject.toml").is_file():
        _root = _root.parent
    sys.path.insert(0, str(_root / "src"))

from blackbox_detection.utils import seed_everything, setup_logger

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import json

from blackbox_detection.stage1.dataset import (
    Stage1ForensicDataset,
    build_dataloader,
    forensic_batch_adapter,
)
from blackbox_detection.stage1.evaluator import (
    AggregationConfig,
    Stage1Evaluator,
    aggregate_unit_predictions,
    evaluate_predictions,
    probabilities_to_labels,
    save_predictions,
    search_best_threshold,
)
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_forensic_samplers
from blackbox_detection.stage1.trainer import Stage1Trainer, TrainConfig
from blackbox_detection.stage1.transforms import (
    PatchAugmentConfig,
    build_forensic_transforms,
)
from blackbox_detection.utils import load_checkpoint, stage1_score

logger = setup_logger("stage1.forensic")

## 2. Paths

In [ ]:
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

CONFIG_DIR = REPO_ROOT / "configs" / "stage1"
OUTPUT_ROOT = REPO_ROOT / "outputs" / "stage1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Team-provided fixed split CSVs.
# Change DATA_DIR / VIDEO_ROOT only if your teammate stores them elsewhere.
DATA_DIR = REPO_ROOT / "data" / "stage1"
TRAIN_CSV = DATA_DIR / "train.csv"
VAL_CSV = DATA_DIR / "val.csv"
TEST_CSV = DATA_DIR / "test.csv"   # reserved for final holdout evaluation; not used for tuning

# Relative video_path values inside CSV are resolved against VIDEO_ROOT.
VIDEO_ROOT = REPO_ROOT

print("repo    :", REPO_ROOT)
print("configs :", CONFIG_DIR)
print("outputs :", OUTPUT_ROOT)
print("train   :", TRAIN_CSV)
print("val     :", VAL_CSV)
print("test    :", TEST_CSV, "(not loaded in training notebooks)")

## 3. Config

In [ ]:
# One of: bayar_resnet18 | chromaticity | frequency | lcdf | cdc
MODEL_NAME = "bayar_resnet18"

FORENSIC_CONFIG = yaml.safe_load(
    (CONFIG_DIR / "forensic.yaml").read_text(encoding="utf-8")
)
if MODEL_NAME not in FORENSIC_CONFIG["models"]:
    raise ValueError(
        f"{MODEL_NAME!r} not in configs/stage1/forensic.yaml; "
        f"available: {sorted(FORENSIC_CONFIG['models'])}"
    )

def merge_config(defaults: dict, overrides: dict) -> dict:
    merged = {key: dict(value) for key, value in defaults.items()}
    for section, values in overrides.items():
        merged.setdefault(section, {})
        merged[section] = {**merged[section], **values}
    return merged

CONFIG = merge_config(
    FORENSIC_CONFIG["defaults"],
    FORENSIC_CONFIG["models"][MODEL_NAME],
)
ADAPTER = forensic_batch_adapter()

SEED = int(CONFIG["train"]["seed"])
PATCH_SIZE = int(CONFIG["data"]["patch_size"])
RUN_DIR = OUTPUT_ROOT / MODEL_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(SEED, deterministic=False)
print(MODEL_NAME, "->", RUN_DIR)
print(json.dumps(CONFIG["model"], indent=2))

## 4. Fixed CSV data

In [ ]:
REQUIRED_COLUMNS = {"video_path", "label", "video_id", "dataset"}
ALLOWED_LABELS = {"ORIGINAL", "RERECORDED"}

def load_stage1_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"CSV not found: {path}")

    frame = pd.read_csv(path)
    missing = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    frame = frame.copy()
    frame["label"] = frame["label"].astype(str).str.strip().str.upper()
    unknown = sorted(set(frame["label"]) - ALLOWED_LABELS)
    if unknown:
        raise ValueError(f"{path.name} has unknown labels: {unknown}")

    def resolve_video_path(value: str) -> str:
        p = Path(str(value))
        if not p.is_absolute():
            p = VIDEO_ROOT / p
        return str(p.resolve())

    frame["video_path"] = frame["video_path"].map(resolve_video_path)
    frame["video_id"] = frame["video_id"].astype(str)
    frame["dataset"] = frame["dataset"].astype(str)

    if frame["video_id"].duplicated().any():
        duplicated = frame.loc[frame["video_id"].duplicated(), "video_id"].head().tolist()
        raise ValueError(f"{path.name} has duplicated video_id values: {duplicated}")

    missing_files = [p for p in frame["video_path"] if not Path(p).is_file()]
    if missing_files:
        raise FileNotFoundError(
            f"{path.name}: {len(missing_files)} video file(s) do not exist. "
            f"First examples: {missing_files[:3]}"
        )
    return frame.reset_index(drop=True)

train_df = load_stage1_csv(TRAIN_CSV)
val_df = load_stage1_csv(VAL_CSV)

overlap = set(train_df["video_id"]) & set(val_df["video_id"])
if overlap:
    raise ValueError(f"train/val video_id leakage detected: {sorted(overlap)[:5]}")

print("train:", len(train_df), train_df["label"].value_counts().to_dict())
print("val  :", len(val_df), val_df["label"].value_counts().to_dict())
print("datasets(train):", train_df["dataset"].value_counts().to_dict())
print("datasets(val)  :", val_df["dataset"].value_counts().to_dict())

### 4.1 Native-resolution frames and patches

In [ ]:
data_config = CONFIG["data"]
augmentation_config = CONFIG["augmentation"]

patch_augment = PatchAugmentConfig(
    hflip_prob=float(augmentation_config["hflip_prob"]),
    vflip_prob=float(augmentation_config["vflip_prob"]),
    rot90_prob=float(augmentation_config["rot90_prob"]),
    spectral_augment_prob=float(augmentation_config["spectral_augment_prob"]),
    spectral_augment_alpha_range=tuple(
        augmentation_config["spectral_augment_alpha_range"]
    ),
    spectral_augment_beta_std=float(
        augmentation_config["spectral_augment_beta_std"]
    ),
    spectral_augment_keep_outside=bool(
        augmentation_config["spectral_augment_keep_outside"]
    ),
)
train_transform, val_transform = build_forensic_transforms(
    train_config=patch_augment
)

train_frame_sampler, train_patch_sampler = build_forensic_samplers(
    train=True,
    num_frames=int(data_config["train_num_frames"]),
    num_patches=int(data_config["train_num_patches"]),
    patch_size=PATCH_SIZE,
)
val_frame_sampler, val_patch_sampler = build_forensic_samplers(
    train=False,
    num_frames=int(data_config["val_num_frames"]),
    num_patches=int(data_config["val_num_patches"]),
    patch_size=PATCH_SIZE,
    val_grid=int(data_config["val_patch_grid"]),
)

train_dataset = Stage1ForensicDataset(
    train_df,
    frame_sampler=train_frame_sampler,
    patch_sampler=train_patch_sampler,
    transform=train_transform,
    patch_size=PATCH_SIZE,
    on_error="zero",
    deterministic=False,
)
val_dataset = Stage1ForensicDataset(
    val_df,
    frame_sampler=val_frame_sampler,
    patch_sampler=val_patch_sampler,
    transform=val_transform,
    patch_size=PATCH_SIZE,
    on_error="zero",
    deterministic=True,
)

train_loader = build_dataloader(
    train_dataset,
    batch_size=int(data_config["batch_size"]),
    shuffle=True,
    num_workers=int(data_config["num_workers"]),
    seed=SEED,
    drop_last=True,
)
val_loader = build_dataloader(
    val_dataset,
    batch_size=int(data_config["val_batch_size"]),
    shuffle=False,
    num_workers=int(data_config["num_workers"]),
    seed=SEED,
)

print("patches per training video:", train_dataset.num_units)
print("patches per validation video:", val_dataset.num_units)
batch = next(iter(train_loader))
adapted = ADAPTER.unpack(batch, "cpu")
print("patch batch:", tuple(batch["patches"].shape), "-> units:", tuple(adapted.inputs.shape))

In [ ]:
first = val_dataset[0]
second = val_dataset[0]
assert torch.equal(first["patches"], second["patches"])
assert first["frame_indices"].tolist() == second["frame_indices"].tolist()
print("deterministic validation patches confirmed")
print("patch range:", float(first["patches"].min()), "-", float(first["patches"].max()))

## 5. Model

In [ ]:
model = build_stage1_model(
    MODEL_NAME,
    finetune_mode=CONFIG["model"]["finetune_mode"],
    unfreeze_last_n=int(CONFIG["model"]["unfreeze_last_n"]),
    **CONFIG["model"]["params"],
)

print("blocks:", len(model.blocks), "| feature dim:", model.feature_dim)
print("parameters:", count_parameters(model))
print("preprocessing:", dict(model.preprocessing()))

with torch.no_grad():
    probe = model(adapted.inputs[:2])
print("logits:", tuple(probe.shape))

## 6. Training

In [ ]:
train_config = CONFIG["train"]

trainer_config = TrainConfig(
    epochs=int(train_config["epochs"]),
    learning_rate=float(train_config["learning_rate"]),
    head_learning_rate=(
        float(train_config["head_learning_rate"])
        if train_config.get("head_learning_rate") is not None
        else None
    ),
    weight_decay=float(train_config["weight_decay"]),
    warmup_ratio=float(train_config["warmup_ratio"]),
    grad_accum_steps=int(train_config["grad_accum_steps"]),
    max_grad_norm=float(train_config["max_grad_norm"]),
    amp=bool(train_config["amp"]),
    label_smoothing=float(train_config.get("label_smoothing", 0.0)),
    early_stopping_patience=int(train_config["early_stopping_patience"]),
    eval_every=int(train_config["eval_every"]),
    seed=SEED,
    output_dir=RUN_DIR,
    model_name=MODEL_NAME,
    wandb_enabled=False,
)

trainer = Stage1Trainer(
    model,
    trainer_config,
    adapter=ADAPTER,
    aggregation=AggregationConfig(
        frame_method=CONFIG["evaluation"]["aggregation"]["frame_method"],
        video_method=CONFIG["evaluation"]["aggregation"]["video_method"],
    ),
    model_config={"name": MODEL_NAME, "params": CONFIG["model"]["params"]},
)

print("device:", trainer.device, "| amp:", trainer.amp)
outcome = trainer.fit(train_loader, val_loader)
print(
    f"best epoch {outcome.best_epoch}: Macro-F1 {outcome.best_macro_f1:.4f} "
    f"at threshold {outcome.best_threshold:.3f}"
)

## 7. Validation

In [ ]:
load_checkpoint(
    RUN_DIR / "best.pt",
    model=model,
    map_location=trainer.device,
    restore_rng_state=False,
)

evaluator = Stage1Evaluator(
    model,
    ADAPTER,
    device=trainer.device,
    amp=trainer.amp,
    aggregation=trainer.aggregation,
)
result, units = evaluator.evaluate(val_loader, return_units=True)

print(f"Macro-F1            : {result.macro_f1:.4f}")
print(f"Macro-F1 @ thr 0.5  : {result.macro_f1_at_default:.4f}")
print(f"optimal threshold   : {result.threshold:.4f}")
print(f"class-wise F1       : {result.per_class_f1}")
print(f"per-dataset Macro-F1: {result.dataset_scores}")
print(f"videos              : {result.num_videos} ({result.num_invalid_videos} with decode problems)")

In [ ]:
rows = []
for video_method in ("mean", "median", "trimmed_mean", "logit_mean", "max"):
    aggregated = aggregate_unit_predictions(
        units,
        aggregation=AggregationConfig(
            frame_method="mean",
            video_method=video_method,
        ),
    )
    scored = evaluate_predictions(aggregated)
    rows.append(
        {
            "frame": "mean",
            "video": video_method,
            "macro_f1": scored.macro_f1,
            "threshold": scored.threshold,
        }
    )
display(pd.DataFrame(rows))

In [ ]:
labels = result.predictions["label"].tolist()
probabilities = result.predictions["prob_rerecorded"].to_numpy()

sweep = pd.DataFrame({"threshold": np.round(np.arange(0.05, 1.0, 0.05), 2)})
sweep["macro_f1"] = [
    stage1_score(labels, probabilities_to_labels(probabilities, threshold))
    for threshold in sweep["threshold"]
]
display(sweep.set_index("threshold").T)

best_threshold, best_score = search_best_threshold(labels, probabilities)
print(f"searched threshold {best_threshold:.4f} -> Macro-F1 {best_score:.4f}")

## 8. Save

In [ ]:
save_predictions(result.predictions, RUN_DIR / "val_predictions.csv")
outcome.history.to_csv(RUN_DIR / "history.csv", index=False)

summary = {
    "model_name": MODEL_NAME,
    "val_macro_f1": float(result.macro_f1),
    "val_macro_f1_at_0.5": float(result.macro_f1_at_default),
    "best_threshold": float(result.threshold),
    "per_class_f1": result.per_class_f1,
    "best_epoch": int(outcome.best_epoch),
    "num_val_videos": int(result.num_videos),
    "preprocessing": dict(model.preprocessing()),
}
(RUN_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2, default=str),
    encoding="utf-8",
)
print("saved to:", RUN_DIR)